## Generating the Large Galaxy 2025 Anchor Set

If you are interested in helping, just claim a statFile currentBin value 

Quillan: 0, 3

You: ?

To do: Ensure bin selection process works and is true

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

from time import perf_counter, sleep
from datetime import datetime

import os
import glob # get path names
from pathlib import Path 
import json

import pandas as pd
from astropy.table import Table, vstack
from astropy.io import fits
import fitsio

from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u

import astropy.visualization as vis # for image enhancements
from astropy.stats import sigma_clip
from scipy.ndimage import gaussian_filter

from SGA.qa import notebook_style, sdss_rgb
from SGA.ssl import load_ssl_embeddings # SGA kernel
from SGA.SGA import get_galaxy_galaxydir, read_sga_sample
import h5py

import ipywidgets as widgets # for buttons
from IPython.display import display, clear_output # for display functions

In [2]:
# Load SGA 2025 set to build anchors from
sample_south, _ = read_sga_sample(region="dr11-south")
_, galaxydirs_south = get_galaxy_galaxydir(sample_south, region="dr11-south")
sample_north, _ = read_sga_sample(region="dr11-north")
_, galaxydirs_north = get_galaxy_galaxydir(sample_north, region="dr11-north")

sample_south["VI_REGION"] = "dr11-south"
sample_north["VI_REGION"] = "dr11-north"

galaxy_lookup = {}
for row, gdir in zip(sample_south, galaxydirs_south):
    galaxy_lookup[(row["VI_REGION"], int(row["SGAID"]))] = {
        "region": "dr11-south",
        "row": row,
        "galaxydir": gdir,
    }
for row, gdir in zip(sample_north, galaxydirs_north):
    galaxy_lookup[(row["VI_REGION"], int(row["SGAID"]))] = {
        "region": "dr11-north",
        "row": row,
        "galaxydir": gdir,
    }

sample = vstack([sample_south, sample_north])
galaxydirs = vstack([galaxydirs_south, galaxydirs_north])

sgaid_to_index = {
    (row["VI_REGION"], int(row["SGAID"])): i
    for i, row in enumerate(sample)
}

INFO:SGA.py:363:_read_catalog: Read 363,633/395,435 GROUP_PRIMARY objects from /dvs_ro/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/sample/SGA2025-beta-v1.6-dr11-south.fits
INFO:SGA.py:370:_read_catalog: Selecting 363,633/363,633 objects in region=dr11-south
INFO:SGA.py:363:_read_catalog: Read 82,093/90,504 GROUP_PRIMARY objects from /dvs_ro/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/sample/SGA2025-beta-v1.6-dr11-north.fits
INFO:SGA.py:370:_read_catalog: Selecting 82,093/82,093 objects in region=dr11-north


In [3]:
galaxydirs

col0
str68
/dvs_ro/cfs/cdirs/cosmo/data/sga/2025/data/dr11-south/000/00006m7218
/dvs_ro/cfs/cdirs/cosmo/data/sga/2025/data/dr11-south/000/00011m0788
/dvs_ro/cfs/cdirs/cosmo/data/sga/2025/data/dr11-south/000/00040m8323
/dvs_ro/cfs/cdirs/cosmo/data/sga/2025/data/dr11-south/000/00070m6618
/dvs_ro/cfs/cdirs/cosmo/data/sga/2025/data/dr11-south/000/00093m8341
/dvs_ro/cfs/cdirs/cosmo/data/sga/2025/data/dr11-south/001/00144p2826
/dvs_ro/cfs/cdirs/cosmo/data/sga/2025/data/dr11-south/001/00158p2020
/dvs_ro/cfs/cdirs/cosmo/data/sga/2025/data/dr11-south/001/00163m1803
/dvs_ro/cfs/cdirs/cosmo/data/sga/2025/data/dr11-south/001/00176m4218


In [6]:
# Diameter constants
PIX_SCALE = 0.262 # arcseconds per pixel
CUTOUT_SIZE_PIX = 152
diamMax = PIX_SCALE * CUTOUT_SIZE_PIX

session_running = True # Session indicator
USERNAME = "Quillan"
rng = np.random.default_rng(seed=67) # DON'T change seed
current_galaxy = None

maybe_mode = False # For uncertain galaxy handling
primary_choice = None

anchor_dir = Path("/pscratch/sd/q/qshimp/VI_training/anchors")
anchor_dir.mkdir(parents=True, exist_ok=True)

# Add classification columns
class_path = Path(f"/pscratch/sd/q/qshimp/VI_training/classifications/{USERNAME}.csv")

if class_path.exists():
    classFile = pd.read_csv(class_path) 
else:
    classFile = pd.DataFrame(columns=[
        "SGAID",
        "Bin", # Diameter bin
        "Morphology",
        "Alt Morph",
        "Main Type", # Out of the four main types
        "Notes"
    ])

# Map main type to morphology
main_type_map = {
    "E": 20,
    "S": 10,
    "L": 0,
    "I": -5
}

stat_path = Path(f"/pscratch/sd/q/qshimp/VI_training/sessions/{USERNAME}.json")

if stat_path.exists():

    with open(stat_path, "r") as f:
        statFile = json.load(f)

else:

    statFile = {
        "current_bin": 0,
        "current_position": 0,
        "total_classified": 0,
        "counts": {
            "E": 0,
            "L": 0,
            "S": 0,
            "I": 0
        }
    }

In [8]:
def binImage(table, diam, min_bin_size=4000):
    diameters = np.asarray(table[diam]) * 60
    max_diam = np.max(diameters)
    n_bins = int(np.ceil(max_diam / diamMax))

    bins = []
    bin_ranges = []
    buffer_idx = []
    buffer_lo = None

    for i in range(n_bins):
        idx = np.where(
            (diameters > i * diamMax) &
            (diameters <= (i + 1) * diamMax)
        )[0]
        if len(idx) == 0:
            continue
        if buffer_lo is None:
            buffer_lo = i * diamMax
        buffer_idx.extend(idx)

        if len(buffer_idx) >= min_bin_size:
            idx_arr = np.array(buffer_idx)
            keys = list(zip(table["VI_REGION"][idx_arr], table["SGAID"][idx_arr]))
            bins.append(keys)
            bin_ranges.append((buffer_lo, (i + 1) * diamMax))
            buffer_idx = []
            buffer_lo = None

    if buffer_idx:
        idx_arr = np.array(buffer_idx)
        keys = list(zip(table["VI_REGION"][idx_arr], table["SGAID"][idx_arr]))
        if bins:
            bins[-1] = bins[-1] + keys
            bin_ranges[-1] = (bin_ranges[-1][0], max_diam)
        else:
            bins.append(keys)
            bin_ranges.append((buffer_lo, max_diam))

    return bins, bin_ranges

In [9]:
# Image enhancer
def stretch_band(band, mode="asinh"):
    band = np.nan_to_num(band)

    vmin = np.nanpercentile(band, 1)
    vmax = np.nanpercentile(band, 99.5)

    if mode == "asinh":
        stretch = vis.AsinhStretch()
    elif mode == "log":
        stretch = vis.LogStretch()
    else:
        stretch = vis.LinearStretch()

    norm = vis.ImageNormalize(
        band,
        interval=vis.ManualInterval(vmin=vmin, vmax=vmax),
        stretch=stretch,
    )

    return norm(band)


def get_galaxy_center(image_path, row):
    with fits.open(image_path) as hdul:
        wcs = WCS(hdul[1].header)

    coord = SkyCoord(row["RA"] * u.deg, row["DEC"] * u.deg)
    x, y = wcs.world_to_pixel(coord)

    return int(round(x)), int(round(y))


def galaxy_center_stretch(image, center, radius=70):
    y, x = np.indices(image.shape)

    mask = (
        (x-center[0])**2 +
        (y-center[1])**2
    ) < radius**2

    values = image[mask]

    vmin = np.percentile(values, 1)
    vmax = np.percentile(values, 99.5)

    norm = vis.ImageNormalize(
        image,
        interval=vis.ManualInterval(vmin, vmax),
        stretch=vis.AsinhStretch()
    )

    return norm(image)


def choose(bands, *choices):
    for band in choices:
        if band in bands:
            return band, bands[band]
    return None, None


# Image loader
def load_image(image_paths, row, stretch="asinh"):

    bands = {
        band: fitsio.read(path).astype(float)
        for band, path in image_paths.items()
        if path is not None
    }

    bandsTitle = "".join(bands.keys())

    g_name, g = choose(bands, "g", "r", "i", "z")
    r_name, r = choose(bands, "r", "i", "g", "z")
    z_name, z = choose(bands, "z", "i", "r", "g")

    try:
        ref_path = image_paths.get("r") or image_paths.get("i") or image_paths.get("z") or image_paths.get("g")
        center = get_galaxy_center(ref_path, row)
    except Exception:
        center = (g.shape[1] // 2, g.shape[0] // 2)

    g_str = galaxy_center_stretch(g, center)
    r_str = galaxy_center_stretch(r, center)
    z_str = galaxy_center_stretch(z, center)

    rgb = None
    lupton = None

    if len(bands) >= 2:
        rgb = np.clip(np.dstack([z_str, r_str, g_str]), 0, 1)

    if {"g", "r", "z"} <= bands.keys():
        lupton = vis.make_lupton_rgb(z, r, g, stretch=0.5, Q=10)
        gray = np.clip(0.5 * r_str + 0.3 * z_str + 0.2 * g_str, 0, 1)
    else:
        _, display_band = choose(bands, "r", "i", "z", "g")
        gray = stretch_band(display_band, stretch)

    smooth = gaussian_filter(gray, sigma=4)

    unsharp = gray - smooth
    unsharp -= unsharp.min()

    if unsharp.max() > 0:
        unsharp /= unsharp.max()

    return rgb, lupton, gray, unsharp, g_str, r_str, z_str, bandsTitle


def load_fits(galaxydir):
    obj_id = os.path.basename(os.path.normpath(galaxydir))
    prefix = f"SGA2025_{obj_id}"

    return {
        band: (
            path
            if os.path.exists(path := os.path.join(galaxydir, f"{prefix}-image-{band}.fits.fz"))
            else None
        )
        for band in ["g", "r", "i", "z"]
    }


def load_jpgs(galaxydir):
    obj_id = os.path.basename(os.path.normpath(galaxydir))
    prefix = f"SGA2025_{obj_id}"

    return {
        name: (
            path
            if os.path.exists(path := os.path.join(galaxydir, f"{prefix}-{name}.jpg"))
            else None
        )
        for name in ["image", "model", "resid"]
    }

In [10]:
# Prepare bins for inspection
bins, bin_ranges = binImage(sample, "D26")
bin_keys = bins[statFile["current_bin"]]
order = rng.permutation(len(bin_keys))
remaining_indices = [bin_keys[i] for i in order]

In [11]:
# Create displays for images and interface
image_out = widgets.Output()
info_out = widgets.Output()

# Create stretch selector
stretch_selector = widgets.Dropdown(
    options=[("Asinh (recommended)", "asinh"), ("Log", "log"), ("Linear", "linear")],
    value="asinh",
    description="Stretch:"
)

# Create view selector
view_selector = widgets.Dropdown(
    options=[
        "RGB",
        "Lupton RGB",
        "Grayscale",
        "Unsharp Mask",
        "RGB + Grayscale",
        "RGB + Grayscale + Unsharp",
        "Image + Model + Residuals",
        "6-view", 
        "g band",
        "r band",
        "z band"
    ],
    value="6-view",
    description="View:"
)

bin_selector = widgets.Dropdown(
    options=[
        (f"Bin {i+1}: {lo:.1f}\"–{hi:.1f}\"", i)
        for i, (lo, hi) in enumerate(bin_ranges)
    ],
    value=statFile["current_bin"],
    description="Diameter bin:"
)

# Create buttons
btn_E = widgets.Button(description="Elliptical")
btn_L = widgets.Button(description="Lenticular")
btn_S = widgets.Button(description="Spiral")
btn_I = widgets.Button(description="Irregular")
btn_maybe = widgets.Button(description="Maybe")
btn_skip = widgets.Button(description="Skip")
btn_stop = widgets.Button(description="Stop")

# Create text box
notes_box = widgets.Textarea(
    value="",
    placeholder="Enter notes here...",
    description="Notes:",
    layout=widgets.Layout(width="600px", height="100px")
)

# Create functional interface
controls = widgets.HBox([bin_selector, stretch_selector, view_selector])
buttons = widgets.HBox([btn_E, btn_L, btn_S, btn_I, btn_maybe, btn_skip, btn_stop])
display(controls, image_out, info_out, buttons, notes_box)

Output()

Output()

Textarea(value='', description='Notes:', layout=Layout(height='100px', width='600px'), placeholder='Enter note…

In [12]:
def show_or_message(image, title, cmap=None):
    plt.title(title)
    plt.axis("off")

    if image is None:
        plt.text(
            0.5, 0.5,
            "Unavailable",
            ha="center",
            va="center",
            transform=plt.gca().transAxes,
            fontsize=12
        )
    else:
        plt.imshow(image, origin="lower")

# Function to display current galaxy
def display_galaxy():

    info = current_galaxy
    row = info["row"]
    gdir = info["galaxydir"]

    paths = load_jpgs(gdir)

    image = mpimg.imread(paths["image"])
    model = mpimg.imread(paths["model"])
    residual = mpimg.imread(paths["resid"])
    
    rgb, lupton, gray, unsharp, g, r, z, bandsTitle = load_image(
    load_fits(info["galaxydir"]),
    info["row"],
    stretch=stretch_selector.value
)
    current_galaxy["bandsTitle"] = bandsTitle

    
    # Display image
    with image_out:

        # Reset image display
        image_out.clear_output(wait=True)
        view = view_selector.value

        # RGB
        if view == "RGB":
            plt.figure(figsize=(6,6))
            show_or_message(rgb, "RGB")

        # Lupton RGB
        elif view == "Lupton RGB":
            plt.figure(figsize=(6,6))
            show_or_message(lupton, "Lupton RGB")

        # Grayscale
        elif view == "Grayscale":

            plt.figure(figsize=(6,6))
            plt.imshow(gray, origin="lower", cmap="gray")
            plt.title("Grayscale")
            plt.axis("off")

        # Unsharp Mask
        elif view == "Unsharp Mask":

            plt.figure(figsize=(6,6))
            plt.imshow(unsharp, origin="lower", cmap="gray")
            plt.title("Unsharp Mask")
            plt.axis("off")

        # RGB + Grayscale
        elif view == "RGB + Grayscale":

            plt.figure(figsize=(10,5))

            plt.subplot(1,2,1)
            show_or_message(rgb, "RGB")

            plt.subplot(1,2,2)
            plt.imshow(gray, origin="lower", cmap="gray")
            plt.title("Grayscale")
            plt.axis("off")

        # RGB + Grayscale + Unsharp
        elif view == "RGB + Grayscale + Unsharp":

            plt.figure(figsize=(15,5))

            plt.subplot(1,3,1)
            show_or_message(rgb, "RGB")

            plt.subplot(1,3,2)
            plt.imshow(gray, origin="lower", cmap="gray")
            plt.title("Grayscale")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(unsharp, origin="lower", cmap="gray")
            plt.title("Unsharp")
            plt.axis("off")

        # Image + Model + Residuals
        elif view == "Image + Model + Residuals":

            plt.figure(figsize=(15,5))

            plt.subplot(1,3,1)
            plt.imshow(image)
            plt.title("Image")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(model)
            plt.title("Model")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(residual)
            plt.title("Residuals")
            plt.axis("off")

        # Image + Model + Residuals + RGB + Grayscale + Unsharp
        elif view == "6-view":

            plt.figure(figsize=(15,10))

            plt.subplot(2,3,1)
            plt.imshow(image)
            plt.title("Image")
            plt.axis("off")

            plt.subplot(2,3,2)
            plt.imshow(model)
            plt.title("Model")
            plt.axis("off")

            plt.subplot(2,3,3)
            plt.imshow(residual)
            plt.title("Residuals")
            plt.axis("off")

            plt.subplot(2,3,4)
            show_or_message(rgb, "RGB")

            plt.subplot(2,3,5)
            plt.imshow(gray, origin="lower", cmap="gray")
            plt.title("Grayscale")
            plt.axis("off")

            plt.subplot(2,3,6)
            plt.imshow(unsharp, origin="lower", cmap="gray")
            plt.title("Unsharp")
            plt.axis("off")
            
        # g band
        elif view == "g band":
            plt.figure(figsize=(6,6))
            show_or_message(g, "g band")

        # r band
        elif view == "r band":
            plt.figure(figsize=(6,6))
            show_or_message(r, "r band")

        # z band
        elif view == "z band":
            plt.figure(figsize=(6,6))
            show_or_message(rgb, "RGB")

        plt.tight_layout()
        plt.show()

In [13]:
# Get new galaxy
def new_galaxy():
    global current_galaxy, remaining_indices, statFile, row
    counts = statFile["counts"]
    current_position = statFile["current_position"]
    current_bin = statFile["current_bin"]

    # Finished current bin?
    if all(v >= 1000 for v in counts.values()) or \
       current_position >= len(remaining_indices):
        print(f"Completed bin {current_bin}. Saving anchors...")
        save_bin_anchors(current_bin)

        print("Moving on...")
        current_bin += 1
        if current_bin >= len(bins):
            print("All galaxies completed!")
            return
        remaining_indices = rng.permutation(bins[current_bin])
        current_position = 0
        statFile["counts"] = {"E": 0, "L": 0, "S": 0, "I": 0}
        statFile["current_bin"] = current_bin
        statFile["current_position"] = current_position
        _sync_bin_selector(current_bin)

    # Get ID for next galaxy
    key = remaining_indices[current_position]  # (region, sgaid)
    current_galaxy = galaxy_lookup[key]
    sgaid = key[1]
    region = key[0]
    row = current_galaxy["row"]
    display_galaxy()
    with info_out:
        info_out.clear_output(wait=True)
        lo, hi = bin_ranges[current_bin]
        print(f"{current_galaxy['bandsTitle']}")
        print(f"Diameter bin: {current_bin + 1}/{len(bins)} ({lo:.1f}\"–{hi:.1f}\")")
        print(f"SGAID: {sgaid}")
        print(f"RA: {row['RA']}")
        print(f"Dec: {row['DEC']}")
        print(f"Diam: {row['D26']*60}\"")
        print(f"Remaining in bin: {len(remaining_indices)-current_position}")
        print()
        print("Classify the galaxy:")

In [14]:
# Handle user input
def handle_answer(choice):
    global statFile, maybe_mode, primary_choice

    uncertain = False
    total = statFile["total_classified"]
    timestamp = datetime.now().isoformat(timespec="seconds")
    
    # Skip if no galaxy
    if current_galaxy is None:
        return

    # Skip galaxy
    if choice == "skip":
        statFile["current_position"] += 1
        save_session()
        new_galaxy()
        return

    # Stop session
    if choice == "stop":
        global session_running
        save_session()

        # Disable buttons
        session_running = False
        ui_enabled(False)
    
        # Enable start button
        btn_stop.description = "Start"
        btn_stop.button_style = "success"
    
        # Pause message
        with info_out:
            info_out.clear_output(wait=True)
            print("Session paused. Click Start to resume.")
    
        return
    
    if choice == "maybe":
    
        maybe_mode = True
        primary_choice = None
    
        with info_out:
            info_out.clear_output(wait=True)
            print("Select the PRIMARY morphology.")
    
        return  

    if maybe_mode:
    
        # First morphology
        if primary_choice is None:
    
            primary_choice = choice
    
            {
                "E": btn_E,
                "L": btn_L,
                "S": btn_S,
                "I": btn_I
            }[choice].disabled = True
    
            with info_out:
                info_out.clear_output(wait=True)
                print(f"Primary morphology: {choice}")
                print("Choose the alternate morphology.")
    
            return
    
        # Second morphology
        morphology = primary_choice
        alt_morph = choice
    
        maybe_mode = False
        primary_choice = None
    
        # Restore all buttons
        for btn in [btn_E, btn_L, btn_S, btn_I]:
            btn.disabled = False
    
    else:
    
        morphology = choice
        alt_morph = ""

    notes = notes_box.value.strip()
    
    main_type = main_type_map[morphology]
    total += 1
    statFile["total_classified"] = total
    statFile["counts"][morphology] += 1
    
    # Append statistics to savefile
    new_row = pd.DataFrame([{
        "SGAID": int(current_galaxy["row"]["SGAID"]),
        "Region": current_galaxy["region"],
        "RA": current_galaxy["row"]["RA"],
        "Dec": current_galaxy["row"]["DEC"],
        "Bin": statFile["current_bin"],
        "Morphology": morphology,
        "Alt Morph": alt_morph,
        "Main Type": main_type,
        "Notes": notes,
        "Timestamp": datetime.now().isoformat(timespec="seconds")
    }])
    
    new_row.to_csv(class_path, mode="a", header=not class_path.exists(), index=False)

    # Display information for user after choice
    with info_out:
        info_out.clear_output(wait=True)
    
    # Keep feedback visible
    sleep(0.5)

    statFile["current_position"] += 1
    save_session()
    notes_box.value = ""
    # Get new galaxy
    new_galaxy()

In [15]:
# Refresh if selector is used
def refresh(change):
    if current_galaxy is not None:
        display_galaxy()
        
stretch_selector.observe(refresh, names="value")
view_selector.observe(refresh, names="value")

# Enable/disable buttons
def ui_enabled(state):
    for btn in [btn_E, btn_L, btn_S, btn_I, btn_maybe, btn_skip]:
        btn.disabled = not state

    stretch_selector.disabled = not state
    view_selector.disabled = not state

# Create stop/start button 
def toggle_session(_):
    global session_running, remaining_indices, statFile, row, current_galaxy

    current_position = statFile["current_position"]
    current_bin = statFile["current_bin"]
    # Stop the session
    if session_running:
        handle_answer("stop")

    # Start the session
    else:
        session_running = True
        ui_enabled(True)

        # Change button from start to stop
        btn_stop.description = "Stop"
        btn_stop.button_style = "danger"

        # Output resume message
        with info_out:
            info_out.clear_output(wait=True)
            print(f"{current_galaxy["bandsTitle"]}")
            print(f"Diameter bin: {current_bin + 1}/{len(bins)}")
            print(f"SGAID: {row["SGAID"]}")
            print(f"RA: {row["RA"]}")
            print(f"Dec: {row["DEC"]}")
            print(f"Diam: {row["D26"]*60}\"")
            print(f"Remaining in bin: {len(remaining_indices)-current_position}")
            print()
            print("Classify the galaxy:")

def save_session():
    with open(stat_path, "w") as f:
        json.dump(statFile, f, indent=4)

def save_bin_anchors(bin_number):
    """
    Save (Region, SGAID) pairs classified in a completed bin.
    """
    if not class_path.exists():
        return
    df = pd.read_csv(class_path)
    bin_df = df[df["Bin"] == bin_number]
    if len(bin_df) == 0:
        print(f"No classifications found for bin {bin_number}")
        return
    anchor_path = anchor_dir / f"{USERNAME}_bin{bin_number}_anchors.csv"
    # Dedupe on the (Region, SGAID) pair, not SGAID alone —
    # otherwise a north/south collision could silently collapse
    # into a single anchor row.
    bin_df[["SGAID", "Region"]].drop_duplicates().to_csv(
        anchor_path,
        index=False
    )
    print(
        f"Saved {len(bin_df)} anchors for bin {bin_number}: {anchor_path}"
    )

def _sync_bin_selector(bin_number):
    bin_selector.unobserve(on_bin_change, names="value")
    bin_selector.value = bin_number
    bin_selector.observe(on_bin_change, names="value")

def set_bin(bin_number, position=0):
    global remaining_indices
    if not (0 <= bin_number < len(bins)):
        raise ValueError(f"Bin must be between 0 and {len(bins)-1}")
    statFile["current_bin"] = bin_number
    statFile["current_position"] = position
    statFile["counts"] = {"E": 0, "L": 0, "S": 0, "I": 0}

    bin_keys = bins[bin_number]  # list of (region, sgaid) tuples
    order = rng.permutation(len(bin_keys))
    remaining_indices = [bin_keys[i] for i in order]

    save_session()
    _sync_bin_selector(bin_number)

def on_bin_change(change):
    if change["name"] != "value" or change["new"] == change["old"]:
        return
    set_bin(change["new"])
    new_galaxy()

bin_selector.observe(on_bin_change, names="value")

In [16]:
# Add button functionality
btn_E.on_click(lambda x: handle_answer("E"))
btn_L.on_click(lambda x: handle_answer("L"))
btn_S.on_click(lambda x: handle_answer("S"))
btn_I.on_click(lambda x: handle_answer("I"))
btn_maybe.on_click(lambda x: handle_answer("maybe"))
btn_skip.on_click(lambda x: handle_answer("skip"))
btn_stop.on_click(toggle_session)

In [17]:
# Initialize session
new_galaxy()

/global/common/software/desi/users/ioannis/SGAML/lib/python3.12/site-packages/astropy/visualization/lupton_rgb.py:645: RuntimeWarning: invalid value encountered in divide
  fInorm = np.where(Int <= 0, 0, np.true_divide(fI, Int))


In [ ]:
'''
import pandas as pd

class_path = Path(f"/pscratch/sd/q/qshimp/VI_training/classifications/{USERNAME}.csv")
df = pd.read_csv(class_path)

# --- Step 1: check for duplicate SGAIDs across different bins ---
dupe_mask = df.duplicated(subset="SGAID", keep=False)
dupes = df[dupe_mask].sort_values("SGAID")

if len(dupes) == 0:
    print("No duplicate SGAIDs found across classifications.")
else:
    # Flag ones that appear under *different* Bin values specifically,
    # since same-bin duplicates could be legitimate re-classifications
    cross_bin_dupes = dupes.groupby("SGAID").filter(lambda g: g["Bin"].nunique() > 1)
    print(f"Found {df.duplicated(subset='SGAID').sum()} duplicate SGAID rows total.")
    print(f"{cross_bin_dupes['SGAID'].nunique()} SGAIDs classified under multiple different bins:")
    print(cross_bin_dupes[["SGAID", "Bin", "Morphology", "Timestamp"]])

# --- Step 2: build fast SGAID -> row lookups per region ---
north_lookup = {int(sgaid): row for sgaid, row in zip(sample_north["SGAID"], sample_north)}
south_lookup = {int(sgaid): row for sgaid, row in zip(sample_south["SGAID"], sample_south)}

def resolve_region_and_coords(sgaid):
    sgaid = int(sgaid)
    # galaxy_lookup always ended up pointing at north on collision,
    # so that's what was actually displayed/classified
    if sgaid in north_lookup:
        row = north_lookup[sgaid]
        return "dr11-north", row["RA"], row["DEC"]
    elif sgaid in south_lookup:
        row = south_lookup[sgaid]
        return "dr11-south", row["RA"], row["DEC"]
    else:
        return None, None, None  # shouldn't happen, but don't silently guess

# --- Step 3: backfill onto the existing dataframe ---
resolved = df["SGAID"].apply(resolve_region_and_coords)
df["Region"] = [r[0] for r in resolved]
df["RA"] = [r[1] for r in resolved]
df["Dec"] = [r[2] for r in resolved]

missing = df["Region"].isna().sum()
if missing:
    print(f"WARNING: {missing} rows had SGAIDs not found in either sample — check these manually.")

# --- Step 4: save backfilled CSV (back up the original first) ---
backup_path = class_path.with_suffix(".csv.bak")
if not backup_path.exists():
    pd.read_csv(class_path).to_csv(backup_path, index=False)
    print(f"Backed up original to {backup_path}")

df.to_csv(class_path, index=False)
print(f"Backfilled Region/RA/Dec for {len(df)} rows.")
'''

In [28]:
rough_anchors = pd.read_csv("/pscratch/sd/q/qshimp/VI_training/classifications/Quillan.csv")
binned_anchors = rough_anchors[rough_anchors['Bin'] == 3]
classes = {g: d for g, d in binned_anchors.groupby('Morphology')}
for g in classes:
    print(g)
    print(len(classes[g]))
print("---------")
subclasses = {g: d for g, d in classes["S"].groupby('Alt Morph')}
for g in subclasses:
    print(g)
    print(len(subclasses[g]))
print("---------")
subclasses = {g: d for g, d in classes["E"].groupby('Alt Morph')}
for g in subclasses:
    print(g)
    print(len(subclasses[g]))
print("---------")
clean_classes = {g: d for g, d in binned_anchors[binned_anchors['Alt Morph'].isna() & binned_anchors['Notes'].isna()].groupby('Morphology')}
for g in clean_classes:
    print(g)
    print(len(clean_classes[g]))

E
233
I
112
L
138
S
550
---------
I
30
L
12
---------
I
1
L
19
---------
E
194
I
90
L
86
S
498


In [30]:
binned_anchors.columns

Index(['SGAID', 'Region', 'RA', 'Dec', 'Bin', 'Morphology', 'Alt Morph',
       'Main Type', 'Notes', 'Timestamp'],
      dtype='str')

In [33]:
rough = binned_anchors.copy()
strict = binned_anchors[
    binned_anchors["Alt Morph"].isna() &
    binned_anchors["Notes"].isna()
].copy()

ANCHOR_DIR = Path(
    "/pscratch/sd/q/qshimp/VI_training/classifications"
)

anchor_columns = [
    "SGAID",
    "Region",
    "RA",
    "Dec",
    "Morphology",
    "Alt Morph",
    "Notes",
]

rough[anchor_columns].to_csv(
    ANCHOR_DIR / "rough_anchors.csv",
    index=False
)

strict[anchor_columns].to_csv(
    ANCHOR_DIR / "strict_anchors.csv",
    index=False
)

print(
    f"Saved {len(rough):,} rough anchors to "
    f"{ANCHOR_DIR / 'rough_anchors.csv'}"
)

print(
    f"Saved {len(strict):,} strict anchors to "
    f"{ANCHOR_DIR / 'strict_anchors.csv'}"
)

Saved 1,033 rough anchors to /pscratch/sd/q/qshimp/VI_training/classifications/rough_anchors.csv
Saved 868 strict anchors to /pscratch/sd/q/qshimp/VI_training/classifications/strict_anchors.csv


In [35]:
anchor_master = binned_anchors[
    anchor_columns
].copy()

rough_keys = set(
    zip(rough["Region"], rough["SGAID"])
)

strict_keys = set(
    zip(strict["Region"], strict["SGAID"])
)

anchor_master["is_rough_anchor"] = [
    (region, sgaid) in rough_keys
    for region, sgaid in zip(
        anchor_master["Region"],
        anchor_master["SGAID"]
    )
]

anchor_master["is_strict_anchor"] = [
    (region, sgaid) in strict_keys
    for region, sgaid in zip(
        anchor_master["Region"],
        anchor_master["SGAID"]
    )
]

anchor_master.to_csv(
    ANCHOR_DIR / "anchor_sets.csv",
    index=False
)

print(
    f"Saved {len(anchor_master):,} anchors to "
    f"{ANCHOR_DIR / 'anchor_sets.csv'}"
)

Saved 1,033 anchors to /pscratch/sd/q/qshimp/VI_training/classifications/anchor_sets.csv


In [37]:
diamMax*4

159.296

In [38]:
diamMax*3

119.472